Qwen Model Training

In [ ]:
# 1. Install Unsloth for 2x faster training and 70% less VRAM usage.
# 2. We install xformers for efficient attention mechanisms.
#!pip uninstall unsloth xformers bitsandbytes peft trl accelerate -y
!pip install unsloth
!pip install --no-deps "xformers<0.0.30" "trl<0.13.0" peft accelerate bitsandbytes

# Restart Session after installation for clean environment

In [ ]:
# Check all versions and CUDA availability to ensure Unsloth compatibility

import torch
import unsloth
import bitsandbytes

print(f"Torch version: {torch.__version__}")
print(f"Unsloth version: {unsloth.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
# This should NOT crash now
from unsloth import FastLanguageModel

In [ ]:
import os
import torch

from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

os.environ["WANDB_DISABLED"] = "true"
torch.cuda.empty_cache()

# 1. Configuration
dataset_path = "dataset_rule_fix/suricata_fixer_dataset.jsonl"
output_dir = "/Suricata-Fixer-Unsloth"
model_id = "unsloth/Qwen2.5-Coder-3B-Instruct"
max_seq_length = 512

# 2. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    device_map = {"": 0},
)

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Load and PRE-TOKENIZE Dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")

def tokenize_function(examples):
    # Standard tokenization logic
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding=False,
    )

print("Pre-tokenizing dataset...")
# FIX: DO NOT remove_columns here. Keep 'text' so the Trainer's internal check passes.
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir = output_dir,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.05,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    # Safely removes the 'text' column once the trainer starts
    remove_unused_columns = True,
)

# 6. Initialize Trainer
# FIX: Point dataset_text_field back to "text" even though we have input_ids
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset,
    dataset_text_field = "text", # Unsloth needs this for its internal 'UnslothSFTTrainer.py' check
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    args = training_args,
    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# 7. Start Training
print(f"Starting Training on {len(tokenized_dataset)} samples...")
trainer.train()

# 8. Save LoRA Adapters
save_path = f"{output_dir}/final-model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Training complete. Saved to: {save_path}")